# Baseline evaluation setup

This notebook prepares the RRUFF spectra for the first fair model benchmark. It chooses which classes to include, creates a held-out test set, and measures a simple majority-class baseline. It does not train PCA or an SVM yet.

Keep the held-out test set untouched while comparing preprocessing and models. If the dataset provides specimen identifiers, split by specimen so spectra from the same specimen cannot appear on both sides.


In [1]:
import numpy as np
import pandas as pd
from raman_data import raman_data
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

# Settings for the first benchmark; record any changes in your results notes.
DATASET_KEY = "rruff_mineral_raw"
MINIMUM_SPECTRA_PER_CLASS = 10
TEST_SIZE = 0.20
RANDOM_STATE = 42

In [2]:
# Load the spectra, labels, and Raman-shift axis from the selected public dataset.
ds = raman_data(DATASET_KEY)
X_all = np.asarray(ds.spectra, dtype=float)
y_all = np.asarray(ds.targets)
wavenumbers = np.asarray(ds.raman_shifts, dtype=float)

# Catch mismatched dimensions before splitting or fitting any model.
if X_all.ndim != 2 or y_all.ndim != 1:
    raise ValueError("Expected 2D spectra and 1D class labels")
if X_all.shape[0] != y_all.shape[0]:
    raise ValueError("There must be one label for every spectrum")
if X_all.shape[1] != len(wavenumbers):
    raise ValueError("Wavenumber axis length must match the number of spectral points")
if not np.all(np.isfinite(X_all)) or pd.isna(y_all).any():
    raise ValueError("Resolve missing or infinite spectra values and missing labels before splitting")

print("Dataset:", DATASET_KEY)
print("Spectra:", X_all.shape[0])
print("Points per spectrum:", X_all.shape[1])
print("Wavenumber range:", wavenumbers.min(), "to", wavenumbers.max(), "cm⁻¹")

2026-09-30 15:21:43,892 INFO raman_data:raman_data:67: Loading dataset: rruff_mineral_raw (cache_dir=None)


Dataset: rruff_mineral_raw
Spectra: 5244
Points per spectrum: 1142
Wavenumber range: 303.2099 to 853.1719000000317 cm⁻¹


In [3]:
# Look for possible metadata fields; verify their meaning before using any as groups.
metadata_candidates = [
    name for name in dir(ds)
    if any(word in name.lower() for word in ("sample", "specimen", "group", "record", "id"))
    and not name.startswith("__")
]
print("Possible metadata fields to inspect:", metadata_candidates)

# Leave as None unless you verify one ID per spectrum from the dataset documentation.
# If you find one, replace None with that aligned array, e.g. np.asarray(ds.<verified_field>).
groups_all = None

# If you assign IDs above, require exactly one non-missing ID per spectrum.
if groups_all is not None:
    groups_all = np.asarray(groups_all)
    if groups_all.ndim != 1 or len(groups_all) != len(y_all) or pd.isna(groups_all).any():
        raise ValueError("Specimen IDs must be a complete 1D array aligned with all spectra")

Possible metadata fields to inspect: ['group_ids']


In [4]:
# Show class sizes from largest to smallest before applying the cutoff.
class_counts_all = pd.Series(y_all, name="mineral").value_counts().rename("spectra")
class_summary = class_counts_all.rename_axis("mineral").reset_index()
display(class_summary.head(25))

eligible_labels = class_counts_all[class_counts_all >= MINIMUM_SPECTRA_PER_CLASS].index
excluded_labels = class_counts_all[class_counts_all < MINIMUM_SPECTRA_PER_CLASS]
print("Classes retained:", len(eligible_labels))
print("Classes excluded for having fewer than", MINIMUM_SPECTRA_PER_CLASS, "spectra:", len(excluded_labels))
if len(eligible_labels) < 2:
    raise ValueError("Fewer than two classes meet the minimum count; lower the cutoff or choose another dataset")

# Apply the same row filter to spectra, labels, and optional specimen IDs.
keep = np.isin(y_all, eligible_labels)
X = X_all[keep]
y = y_all[keep]
groups = None if groups_all is None else np.asarray(groups_all)[keep]
print("Spectra retained:", len(y), "of", len(y_all))

,mineral,spectra
0,637,40
1,407,30
2,458,29
3,55,27
4,164,26
5,245,26
6,550,25
7,29,24
8,415,23
9,446,23


Classes retained: 66
Classes excluded for having fewer than 10 spectra: 1619
Spectra retained: 1045 of 5244


In [5]:
all_labels = np.unique(y)
indices = np.arange(len(y))

if groups is None:
    # Stratify by mineral so each eligible class is represented on both sides.
    train_idx, test_idx = train_test_split(
        indices,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )
    split_description = "stratified by mineral label"
else:
    # Keep spectra from one physical specimen together to reduce sample-level leakage.
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))
    split_description = "grouped by verified specimen ID"

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# A class absent from training cannot be learned by the later classifier.
missing_from_train = np.setdiff1d(all_labels, np.unique(y_train))
if len(missing_from_train):
    raise ValueError(
        f"These classes are absent from training after the split: {missing_from_train}. "
        "For grouped splits, try another seed or reconsider the class cutoff."
    )

print("Split method:", split_description)
print("Training spectra:", len(y_train))
print("Held-out test spectra:", len(y_test))
if groups is not None:
    overlap = np.intersect1d(groups[train_idx], groups[test_idx])
    if len(overlap):
        raise ValueError("A specimen ID appears in both train and test")
    print("Specimen IDs overlap:", False)
else:
    print("Specimen IDs overlap: not checked (no verified IDs available)")

missing_from_test = np.setdiff1d(all_labels, np.unique(y_test))
if len(missing_from_test):
    print("WARNING: some classes are absent from test; review the split before interpreting scores:", missing_from_test)

# Compare proportions so a grouped split's class imbalance is visible.
train_share = pd.Series(y_train).value_counts(normalize=True).rename("train_share")
test_share = pd.Series(y_test).value_counts(normalize=True).rename("test_share")
split_counts = pd.concat([train_share, test_share], axis=1).fillna(0)
display(split_counts.sort_values("test_share").head(20))

Split method: stratified by mineral label
Training spectra: 836
Held-out test spectra: 209
Specimen IDs overlap: not checked (no verified IDs available)


,train_share,test_share
53,0.010766,0.009569
1208,0.010766,0.009569
183,0.010766,0.009569
533,0.010766,0.009569
740,0.009569,0.009569
1103,0.009569,0.009569
49,0.009569,0.009569
662,0.009569,0.009569
1513,0.009569,0.009569
1681,0.009569,0.009569


In [6]:
# Fit only on training labels; the test labels are used only for scoring.
majority_model = DummyClassifier(strategy="most_frequent")
majority_model.fit(X_train, y_train)
majority_predictions = majority_model.predict(X_test)

baseline_scores = pd.Series({
    "accuracy": accuracy_score(y_test, majority_predictions),
    "balanced accuracy": balanced_accuracy_score(y_test, majority_predictions),
    "macro F1": f1_score(
        y_test,
        majority_predictions,
        labels=all_labels,
        average="macro",
        zero_division=0
    ),
})
print("Majority class predicted:", majority_model.classes_[0])
display(baseline_scores.to_frame("majority baseline"))

Majority class predicted: 2


,majority baseline
accuracy,0.038278
balanced accuracy,0.015152
macro F1,0.001117
